# 01 · Meet the local models
Run the API examples here in Jupyter. Then give OpenClaw the same job and watch it use tools.

**Lightweight orchestration: Nano. Task reasoning when needed: Lightning.**

| Component | Job |
|---|---|
| **OpenClaw + Nemotron Nano** | The agent harness uses the lightweight Nano model to read the skill, choose tools, and coordinate the steps. |
| **Tools + Nemotron Lightning** | Tools call Lightning when the task needs its reasoning: use `base` for the general model or `specialist` for the model with our trained adapter. |
| **OpenShell** | Runs OpenClaw and its tools inside the sandbox. |

The OpenClaw terminal displays **Nano**, its orchestration model. The API examples below call **Lightning**, the model used by the tools.

This is a live walkthrough. Requests use the existing services; opening the notebook does not execute them.

###  model APIs and agent tools

[OpenClaw skills](https://docs.openclaw.ai/tools/skills) explains how an agent discovers instructions. For serving, the [vLLM LoRA guide](https://docs.vllm.ai/en/latest/features/lora/) explains separately named base/adapter endpoints. These are current upstream guides; our tested runtime versions are in the [SDK map](README.md#concept-sdks-and-documentation).

verify `base` and `specialist` on `http://127.0.0.1:8002/v1/models`, then run the two requests below. The manual exercise needs only that API. The agent handoff additionally needs Nano on `30081`, the OpenShell sandbox, and local checkpoint/adapter metadata; follow [service setup](../README.md#provision-services-before-running-the-course).

**What to inspect:** the requested model, returned model, visible answer, finish reason, and latency. A model-list entry is not proof of the loaded checkpoint revision or classification quality.

Source: [common.py](common.py) sends HTTP requests; [inventory_tool.py](agent/inventory_tool.py) implements the five inventory actions. The OpenClaw terminal is controlled by Nano, while its probe tools call Lightning. The workshop currently uses local Chat Completions; an OpenAI-hosted backend is not enabled by changing the URL alone.


### Start here
Load the small HTTP helper. It sends requests to the local model server.

In [ ]:
from pathlib import Path
import sys
from time import perf_counter
from IPython.display import JSON, Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.common import request, BASE_URL
print("Model server:", BASE_URL)

### 1 · Which models are available?
`GET /v1/models` asks the server for its model list. Run this cell to see the **live JSON for each model**.

`id` is the name we use in a request. `root` is the location reported by the server. The specialist’s `parent` links it to the base model.

In [ ]:
models = request("/v1/models")
for model in models["data"]:
    print(model["id"])
    display(JSON(model, expanded=True))

### 2 · Ask the base model
`POST /v1/chat/completions` sends a message. First inspect the request, then send it.

In [ ]:
base_request = {
    "model": "base",
    "messages": [{"role": "user", "content": "Reply with one short sentence acknowledging this endpoint probe."}],
    "max_tokens": 128,
    "temperature": 0,
    "chat_template_kwargs": {"enable_thinking": False},
}
display(JSON(base_request, expanded=True))

In [ ]:
started = perf_counter()
base_response = request("/v1/chat/completions", base_request)
print(f"Base request: {perf_counter() - started:.3f} seconds")
display(JSON(base_response, expanded=True))

### 3 · Ask the specialist
Same message; change only `model` to `specialist`. Run these cells to see its request and fresh response.

In [ ]:
specialist_request = {**base_request, "model": "specialist"}
display(JSON(specialist_request, expanded=True))

In [ ]:
started = perf_counter()
specialist_response = request("/v1/chat/completions", specialist_request)
print(f"Specialist request: {perf_counter() - started:.3f} seconds")
display(JSON(specialist_response, expanded=True))

**Read the response:** `message.content` is the answer. `finish_reason: "stop"` means generation finished; `length` means it hit the output limit.

This checks that both endpoints answer. It does not measure classification accuracy. API roots also do not prove which cached checkpoint revision is loaded.

### 4 · Give the agent a playbook
**You → OpenClaw + lightweight Nano → tools → Lightning when needed → evidence**

Nano coordinates the work: read the skill, select a tool, and use its result to decide the next step. Tools handle direct checks themselves and call Lightning when model reasoning is needed. The specialist adds our trained adapter to Lightning.

In this notebook, the Lightning calls are simple endpoint probes. They show the connection between orchestration and model calls; they do not demonstrate reasoning quality.

| Tool action | What you will see |
|---|---|
| `inventory` | Host-prepared checkpoint and adapter metadata |
| `models` | The live model list |
| `probe-base` | A fresh base request |
| `probe-specialist` | A fresh specialist request |
| `manifest` | Evidence and validation status |

Open the actual skill below. These are the instructions the agent will read.

In [ ]:
from html import escape
from IPython.display import HTML
from course.agent.notebook_terminal import prepare_terminal, open_terminal, collect_terminal

skill = (ROOT / "course/agent/skills/workshop-model-inventory/SKILL.md").read_text()
display(HTML("<details><summary><b>Read SKILL.md</b></summary><pre style='white-space:pre-wrap'>"
             + escape(skill) + "</pre></details>"))

### 5 · Prepare OpenClaw
Run once for each new demo. This cell copies the skill and tool into a fresh sandbox run directory and snapshots the host’s model metadata. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. It does not train or modify the model. The source is `course/agent/skills/workshop-model-inventory/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox:

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── inventory_tool.py             # Python helper invoked by the agent
├── host-inventory.json           # exported host metadata snapshot
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        └── workshop-model-inventory/
            └── SKILL.md          # installed skill instructions
```

OpenClaw discovers the skill under `workspace/skills/`. The task you paste in the next step explicitly tells the agent to read this skill and supplies the full path to `inventory_tool.py` (the skill’s `<tool_path>` placeholder). The agent then uses its execution tool to run the helper commands.

The cell prints the actual sandbox paths for this run. The local run directory stores collected results on the host; the installed skill lives at the sandbox path shown below.


In [ ]:
terminal_run = prepare_terminal()
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-model-inventory/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/inventory_tool.py")

### 6 · Open the TUI and paste the task
The next cell opens a terminal **on this workshop computer’s desktop**. Exit any other OpenClaw TUI with `/exit` first.

The TUI’s model label will say **Nemotron Nano**: that is the lightweight orchestration model. Its tools will call the **Lightning base and specialist** endpoints.

Copy only the task printed below into OpenClaw. Watch it read the skill and run the five tool actions. The agent makes its own fresh requests; the examples above are separate.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### 7 · Bring the evidence back
Wait until OpenClaw finishes, then run this cell. It retrieves **this run’s** manifest and checks the tool trace, including whether the skill was read before execution.

If you use **Run All** before completing the TUI task, this cell will stop with “No manifest yet.” Run it again after the agent finishes.

In [ ]:
validation = collect_terminal(terminal_run)
display(JSON(validation, expanded=True))
import json
manifest = json.loads(Path(validation["manifest"]).read_text())
display(JSON(manifest, expanded=False))

**What you demonstrated:** call the API yourself → give an agent a skill → watch it call tools → verify its evidence.

The TUI can stay open while you retrieve the results. Use `/exit` when done. Run the preparation cell again only when you want a new independent run.